## A1.1 User–Item Matrix Sparsity and Matrix Factorization

**Theoretical answer**

A user–item interaction matrix $R$ of size $2{,}500{,}000 \times 400{,}000$ is extremely sparse because each user interacts with only a tiny fraction of the full catalog. In an e-commerce platform, a typical user may browse, click, or purchase tens to hundreds of products, while the total catalog contains hundreds of thousands of SKUs. Therefore, most entries in $R$ are 0, meaning 'no interaction'.

Matrix factorization approximates the sparse matrix as:

$$R \approx U V^T$$

where:
- $U$ is the **user latent factor matrix**
- $V$ is the **item latent factor matrix**
- each user and item is represented in a smaller latent space

If the number of latent factors is $k = 50$:
- $U$ has shape **(2,500,000 × 50)**
- $V$ has shape **(400,000 × 50)**

This reduces dimensionality and helps predict missing interactions, which is useful in recommendation systems.

In [1]:
import numpy as np

# Example of user-item interaction matrix
R_demo = np.array([
    [1, 0, 0, 1, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1],
])

total_entries = R_demo.size
non_zero_entries = np.count_nonzero(R_demo)
zero_entries = total_entries - non_zero_entries
sparsity = zero_entries / total_entries

print('Demo interaction matrix:\n', R_demo)
print(f'Total entries: {total_entries}')
print(f'Non-zero entries: {non_zero_entries}')
print(f'Zero entries: {zero_entries}')
print(f'Sparsity: {sparsity:.2%}')

Demo interaction matrix:
 [[1 0 0 1 0]
 [0 0 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 1]]
Total entries: 20
Non-zero entries: 6
Zero entries: 14
Sparsity: 70.00%


## A1.2 Cosine Similarity Between Two Users

**Problem**

User A = [1, 0, 1, 1, 0, 1, 0]  
User B = [1, 1, 0, 1, 0, 0, 1]

**Theoretical answer**

Cosine similarity is:

$$\cos(\theta) = \frac{A \cdot B}{\|A\|\|B\|}$$

Step-by-step:
- Dot product: $1\cdot1 + 0\cdot1 + 1\cdot0 + 1\cdot1 + 0\cdot0 + 1\cdot0 + 0\cdot1 = 2$
- $\|A\| = \sqrt{1^2+0^2+1^2+1^2+0^2+1^2+0^2} = \sqrt{4} = 2$
- $\|B\| = \sqrt{1^2+1^2+0^2+1^2+0^2+0^2+1^2} = \sqrt{4} = 2$

Therefore:

$$\cos(\theta) = \frac{2}{2\times2} = 0.5$$

A cosine similarity of **0.5** indicates moderate similarity. I would not blindly recommend exactly the same products to both users. Instead, I would treat them as partially similar and combine this signal with product content, recency, and browsing behavior.

In [2]:
import numpy as np

user_a = np.array([1, 0, 1, 1, 0, 1, 0])
user_b = np.array([1, 1, 0, 1, 0, 0, 1])

dot_product = np.dot(user_a, user_b)
norm_a = np.linalg.norm(user_a)
norm_b = np.linalg.norm(user_b)
cosine_similarity = dot_product / (norm_a * norm_b)

print('User A:', user_a)
print('User B:', user_b)
print('Dot product:', dot_product)
print('Norm of A:', norm_a)
print('Norm of B:', norm_b)
print('Cosine similarity:', round(cosine_similarity, 4))

User A: [1 0 1 1 0 1 0]
User B: [1 1 0 1 0 0 1]
Dot product: 2
Norm of A: 2.0
Norm of B: 2.0
Cosine similarity: 0.5


## A1.3 PCA and Explained Variance

**Problem**

Top 5 explained variance ratios are:
[0.38, 0.22, 0.14, 0.09, 0.07]

**Theoretical answer**

Cumulative explained variance:
- Component 1: 0.38
- Component 2: 0.60
- Component 3: 0.74
- Component 4: 0.83
- Component 5: 0.90

So the first 5 components explain **90%** of the variance, not 95%. Therefore, more than 5 components are needed to reach 95% variance. Based on the remaining variance, the final number would likely be around **6–8 components**, depending on the next ratios.

I would prefer PCA over manual feature selection when:
- there are many correlated features
- the raw feature space is high-dimensional
- I want a compact representation with less noise
- computational efficiency matters

For recommendation models, PCA is useful when many engineered features overlap or when dimensionality reduction improves speed without losing much information.

In [3]:
import numpy as np

explained_variance_ratio = np.array([0.38, 0.22, 0.14, 0.09, 0.07])
cumulative_variance = np.cumsum(explained_variance_ratio)

for i, value in enumerate(cumulative_variance, start=1):
    print(f'Components {i}: cumulative explained variance = {value:.2f}')

reaches_95 = np.where(cumulative_variance >= 0.95)[0]
if len(reaches_95) > 0:
    print('Minimum components to reach 95% variance:', reaches_95[0] + 1)
else:
    print('Top 5 components only explain 90%, so more than 5 components are needed.')


Components 1: cumulative explained variance = 0.38
Components 2: cumulative explained variance = 0.60
Components 3: cumulative explained variance = 0.74
Components 4: cumulative explained variance = 0.83
Components 5: cumulative explained variance = 0.90
Top 5 components only explain 90%, so more than 5 components are needed.
